# Retrain v3 — curated bridge/concrete mix

Trains **your custom U-Net** (SE blocks, bottleneck dropout, deep supervision) on UAV Kaggle + DeepCrack + Auto-ROS-LAB UAV 11k. Run cells top to bottom.

**Before you start (upload to MyDrive):**
- `crack-seg_revision_8-2.zip` — the repo code (get it from the project folder; if you've pushed `revision_8-2` to GitHub, you can skip this and the notebook will clone).
- `kaggle.json` — for the UAV dataset download (kaggle.com → Settings → API → Create New Token). Required unless you re-upload `dataset_split.zip` to `MyDrive/bridge_crack_detection/` to keep the original 220/47/48 split.
- DeepCrack's license is non-commercial research/educational — fine for this project.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/crack_v3'
DATA_ROOT = '/content/crack_data'
OUT = DATA_ROOT + '/dataset_split'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)
print('Data out :', OUT)

## 2. Get the code + dependencies

In [ ]:
import os, sys, shutil
ZIP = '/content/drive/MyDrive/crack-seg_revision_8-2.zip'
if os.path.isdir('/content/crack-seg'):
    REPO = '/content/crack-seg'
elif os.path.isdir('/content/src') and os.path.isdir('/content/scripts'):
    REPO = '/content'
else:
    REPO = '/content/crack-seg'
    if os.path.isdir('/content/drive/MyDrive/crack-seg'):
        shutil.copytree('/content/drive/MyDrive/crack-seg', REPO)
        print('Copied repo from Drive folder.')
    elif os.path.exists(ZIP):
        shutil.unpack_archive(ZIP, '/content')
        if not os.path.isdir(REPO) and os.path.isdir('/content/src'):
            REPO = '/content'
        print('Unzipped repo from Drive.')
    else:
        print('Cloning revision_8-2 (push it to GitHub first, or upload crack-seg_revision_8-2.zip to MyDrive).')
        os.system('git clone -b revision_8-2 https://github.com/Ishaan1402/crack-seg.git ' + REPO)
os.chdir(REPO)
sys.path.insert(0, REPO)
os.system('pip install -q -r requirements.txt kagglehub gdown')
print('Setup complete in', os.getcwd())

### Kaggle credentials (required for the UAV source)

In [ ]:
from scripts import colab_data as cd
KAGGLE_JSON = '/content/drive/MyDrive/kaggle.json'
if cd.setup_kaggle_credentials(KAGGLE_JSON):
    print('Kaggle credentials loaded from MyDrive.')
else:
    print('WARNING: kaggle.json not found in MyDrive — the UAV download will fail')
    print('unless you re-upload dataset_split.zip. Get kaggle.json from')
    print('kaggle.com -> Settings -> API -> Create New Token and upload it to MyDrive.')

## 3. Download & stage sources

### UAV Kaggle (primary — fresh 70/15/15 split via Kaggle, or original split if dataset_split.zip is on Drive)

In [ ]:
from scripts import colab_data as cd
DRIVE_ZIP = '/content/drive/MyDrive/bridge_crack_detection/dataset_split.zip'
mode = cd.download_uav(DRIVE_ZIP if os.path.exists(DRIVE_ZIP) else None, DATA_ROOT, OUT)
print('UAV staged from:', mode, '(drive = original 220/47/48 split, kaggle = fresh 70/15/15)')

### DeepCrack (train only; its test set is held out for evaluation)

In [ ]:
dc_train_img, dc_train_lab, DC_TEST_IMG, DC_TEST_LAB = cd.download_deepcrack(DATA_ROOT)
cd.stage(dc_train_img, dc_train_lab, OUT, source='deepcrack', cap=300, resize=512, val_frac=0.1, test_frac=0.0, seed=42)
print('DeepCrack test held out for eval:', DC_TEST_IMG)
print('                                 ', DC_TEST_LAB)

### Auto-ROS-LAB UAV 11k

In [ ]:
uav11k_images, uav11k_masks = cd.download_uav11k(DATA_ROOT)
cd.stage(uav11k_images, uav11k_masks, OUT, source='uav11k', cap=8000, resize=512, val_frac=0.1, test_frac=0.05, seed=42)

## 4. Staging summary (per-source sanity check)

In [ ]:
import json
with open(OUT + '/manifest.json') as f:
    manifest = json.load(f)
print(f"{'source':10s} {'train':>7s} {'val':>7s} {'test':>7s} {'crack%':>8s}")
for src, m in manifest.items():
    print(f"{src:10s} {m['train']:7d} {m['val']:7d} {m['test']:7d} {m['mean_crack_ratio']*100:7.2f}%")

## 5. Train v3 (your U-Net, upgraded)

- Primary run: narrow `[32,64,128,256]` + SE + dropout 0.1 + deep supervision, 512px, strong aug, AMP.
- Optional wide comparison: set `RUN_WIDE = True` and rerun this cell.
- Hardware presets auto-select batch size (T4/L4 vs A100/V100).
- If the session dies, rerun with `--checkpoint` pointing at the last `.pth` to resume from those weights.

In [ ]:
from scripts import train as train_mod
RUN_WIDE = False
features = '64,128,256,512' if RUN_WIDE else '32,64,128,256'
out_name = 'unet_v3_wide.pth' if RUN_WIDE else 'unet_v3_narrow.pth'
out_path = DATA_ROOT + '/' + out_name
train_mod.main([
    '--data-dir', OUT,
    '--out', out_path,
    '--epochs', '30',
    '--lr', '1e-3',
    '--resize', '512',
    '--features', features,
    '--se', '--dropout', '0.1', '--deep-supervision',
    '--aug', 'strong',
    '--amp',
])

## 6. Save to Drive + evaluation commands

In [ ]:
import shutil
shutil.copy(out_path, DRIVE_ROOT + '/' + out_name)
print('Saved to Drive:', DRIVE_ROOT + '/' + out_name)
print()
print('HF upload (after huggingface-cli login):')
print(f'  huggingface-cli upload ishaan1402/crack-seg {out_path} unet_v3.pth')
print()
print('Cross-domain eval on DeepCrack test:')
print(f'  PYTHONPATH={REPO} python scripts/verify_metrics.py --checkpoint {out_path} --images {DC_TEST_IMG} --masks {DC_TEST_LAB} --mode both --thresholds 0.3 0.4 0.5 0.6 0.7')
print()
print('In-distribution eval on the staged test split (all sources, untouched during training):')
print(f'  PYTHONPATH={REPO} python scripts/verify_metrics.py --checkpoint {out_path} --images {OUT}/test/images --masks {OUT}/test/masks --mode both --thresholds 0.5')
print('  (files named uav_* in that split are the UAV Kaggle subset)')

## Notes

- The staged dataset lives in Colab's ephemeral disk; only checkpoints are copied to Drive. Re-running the download cells after a session reset is expected.
- The UAV source now gets a fresh 70/15/15 split (your old dataset_split is gone). Treat that split as the fixed UAV test set going forward and do not regenerate it between runs.
- `manifest.json` records per-source counts + mean crack ratio — a source with a suspiciously low/high crack% is a red flag worth inspecting before trusting the run.
- After training, update the HF model card with test-set numbers only (run `verify_metrics.py` on the real UAV test split, not just validation).